# RDMatcher example (Gower default)

This notebook demonstrates a minimal end-to-end example using the RDMatcher API with Gower distance. It generates a synthetic cohort, initializes the matcher, (optionally) computes propensity logits, runs rare matching, and shows simple diagnostics and plots.


In [ ]:
import sys
# If running from repository checkout, make sure src is on the path
sys.path.insert(0, 'src')

import pandas as pd
import matplotlib.pyplot as plt

from rdmatcher import RDMatcher
from rdmatcher.utils import make_cohort_independent as make_cohort
from rdmatcher.plot import plot_feature_balance, plot_propensity_support

%matplotlib inline


In [ ]:
# --- Synthetic cohort parameters (small for demo) ---
npats = 10000
ncases = 100
seed_value = 404

# Categorical distributions for controls and cases
control_sex = {'Female': 0.52, 'Male': 0.46, 'Nonbinary': 0.01, 'Unknown': 0.01}
case_sex = {'Female': 0.56, 'Male': 0.41, 'Nonbinary': 0.02, 'Unknown': 0.01}

control_re = {
    'Asian': 0.07,
    'Black or African American': 0.05,
    'Native American or Alaska Native': 0.02,
    'Native Hawaiian or Other Pacific Islander': 0.03,
    'Other': 0.14,
    'Unknown/Declined': 0.33,
    'White': 0.36
}
case_re = {
    'Asian': 0.09,
    'Black or African American': 0.06,
    'Native American or Alaska Native': 0.01,
    'Native Hawaiian or Other Pacific Islander': 0.01,
    'Other': 0.16,
    'Unknown/Declined': 0.30,
    'White': 0.37
}

# Feature definitions: for continuous features we specify (distribution_name, [params])
features = {
    'age_at_first_visit': ('continuous', ('normal', [32, 12]), ('normal', [38, 5])),
    'person_time': ('continuous', ('normal', [7, 7]), ('normal', [9, 3])),
    'encounters': ('continuous', ('normal', [20, 10]), ('normal', [25, 9])),
    'sex': ('categorical', control_sex, case_sex),
    'race_ethnicity': ('categorical', control_re, case_re)
}

# Generate the cohort
cohort = make_cohort(npats, ncases, features, seed=seed_value)

# (Optional) introduce modest missingness for realism (10%)
for col in cohort.columns:
    if col not in ['patient_id', 'exposure_status']:
        cohort.loc[cohort.sample(frac=0.1, random_state=seed_value).index, col] = None

cohort.head()


In [ ]:
# Initialize the RDMatcher with Gower defaults (no one-hot encoding)
matcher = RDMatcher(
    pop_df=cohort,
    patient_id_col='patient_id',
    exposure_status='exposure_status',
    features_numeric=['age_at_first_visit', 'person_time', 'encounters'],
    features_categorical=['sex', 'race_ethnicity'],
    process_features=False,
    onehot=False,
    debug=False
)

# (Optional) Compute propensity logits — useful when using method='propensity'
# matcher.calculate_propensity_logits(method='simple')

# Run matching (Gower) — return matched data for immediate inspection
matched_df = matcher.rare_matching(
    threshold=0.1,
    n_neighbors=1,
    k_candidates=500,
    method='multi',
    distance_metric='gower',
    global_optimal=True,
    replacement=False,
    competitive_match=True,
    diagnostics=True,
    return_matched_data=True
)

matched_df.head()


In [ ]:
# Show a brief diagnostics summary and plots
print('Summary table (top rows):')
matcher.summary_table

In [ ]:

# Propensity coverage before/after matching
# matcher.plot_propensity_coverage(compare_matching=True, figsize=(10,4))


# Plot feature balance for a numeric feature
plot_feature_balance(cohort_before=matcher.pop, cohort_after=matcher.pop_matched, feature='age_at_first_visit', exposure_status_col='exposure_status', normalize=True)


In [ ]:
# -----------------------------
# Propensity score examples
# -----------------------------

# 1) Fit a propensity model (formula-driven)
matcher.fit_propensity_model(formula='age_at_first_visit + sex + race_ethnicity', random_state=404)
matcher.logger.info('<system-reminder> propensity_logit added to data: {}'.format('propensity_logit' in matcher.pop.columns))
matched_propensity = matcher.rare_matching(threshold=0.1, n_neighbors=1, method='propensity', distance_metric='euclidean', return_matched_data=True)
matched_propensity.head()


## PSM + RDM Hybrid (Propensity Caliper + Gower Matching)

The recommended two-step approach (`ps_hybrid=True`):

1. **Fit a propensity model** to estimate each subject's probability of treatment.
2. **Apply a PS caliper** — for each treated subject, only controls within a propensity-based distance are eligible.
3. **Match on covariates using Gower distance** within the eligible set.

This prevents poor matches that are far apart on the propensity scale while still leveraging Gower's ability to handle mixed data types and missing values. Propensity is used *only* for filtering — it does **not** enter the Gower distance calculation, avoiding double-counting.

In [ ]:
# Re-initialize matcher (fresh start for hybrid demo)
matcher_hybrid = RDMatcher(
    pop_df=cohort,
    patient_id_col='patient_id',
    exposure_status='exposure_status',
    features_numeric=['age_at_first_visit', 'person_time', 'encounters'],
    features_categorical=['sex', 'race_ethnicity'],
    process_features=False,
    onehot=False,
    debug=False
)

# Step 1: Fit propensity model
matcher_hybrid.fit_propensity_model(
    formula='age_at_first_visit + person_time + encounters + sex + race_ethnicity',
    random_state=404
)

# Step 2: Match with PS caliper + Gower (ps_hybrid=True)
matched_hybrid = matcher_hybrid.rare_matching(
    threshold=0.1,
    n_neighbors=1,
    k_candidates=500,
    method='multi',
    distance_metric='gower',
    global_optimal=True,
    competitive_match=True,
    ps_hybrid=True,
    ps_caliper=0.2,
    ps_caliper_strict=True,
    diagnostics=True,
    return_matched_data=True
)

matched_hybrid.head()

In [ ]:
# Propensity coverage: compare before and after hybrid matching
matcher_hybrid.plot_propensity_coverage(compare_matching=True)

## Mahalanobis distance (numeric-only)

RDMatcher also supports standard Mahalanobis distance for numeric-only covariates. It uses the square-root Mahalanobis distance, so `threshold` is compared directly against that value. Missing values are handled with complete-case behavior on the fitted reference rows; queries containing NaN return NaN distances.

In [ ]:
matcher_maha = RDMatcher(
    pop_df=cohort,
    patient_id_col='patient_id',
    exposure_status='exposure_status',
    features_numeric=['age_at_first_visit', 'person_time', 'encounters'],
    features_categorical=['sex', 'race_ethnicity'],
    process_features=False,
    onehot=True,
)

matched_maha = matcher_maha.rare_matching(
    threshold=2.0,
    n_neighbors=1,
    k_candidates=200,
    method='multi',
    distance_metric='mahalanobis',
    global_optimal=True,
    competitive_match=True,
    return_matched_data=True,
)

matched_maha.head()